# MI-GAN Distillation Training

Distill LaMa inpainting (198MB) → MI-GAN student (~16MB) for CoreML.

**Setup**: Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Pro)

| Step | T4 (free) | A100 (Pro) |
|------|-----------|------------|
| Cache generation | ~60 min | ~15 min |
| Training 500 kimg | ~3-4 hr | ~1 hr |

In [ ]:
# Step 1: Setup
!git clone https://github.com/nghyane/Typoon.git
%cd Typoon/scripts/distill
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q onnxruntime-gpu onnx Pillow numpy tensorboard tqdm huggingface_hub

import torch
assert torch.cuda.is_available(), 'No GPU! Change runtime type.'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu} ({vram:.0f}GB VRAM)')

In [ ]:
# Step 2: Download data from HuggingFace Hub
from huggingface_hub import hf_hub_download
import zipfile, os

REPO = 'nghyane/migan-training-data'
os.makedirs('../../data/training', exist_ok=True)
os.makedirs('../../models', exist_ok=True)

for fname, dest in [
    ('training_manga.zip', '../../data/training/'),
    ('training_manhwa.zip', '../../data/training/'),
    ('lama_fp32.zip', '../../models/'),
]:
    print(f'Downloading {fname}...')
    path = hf_hub_download(repo_id=REPO, filename=fname, repo_type='dataset')
    print(f'Extracting {fname}...')
    with zipfile.ZipFile(path, 'r') as z:
        z.extractall(dest)

n_images = int(os.popen('find ../../data/training -type f | wc -l').read().strip())
print(f'Done! {n_images} training images')

In [ ]:
# Step 3: Generate teacher cache (~30-60 min)
!python generate_teacher_cache.py \
    --data_dir ../../data/training \
    --output_dir ../../data/teacher_cache \
    --lama_model ../../models/lama_fp32.onnx \
    --masks_per_image 5 \
    --batch_size 4 \
    --random_crop

In [ ]:
# Step 4: Train MI-GAN
import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
BATCH = 32 if vram >= 30 else 16 if vram >= 14 else 8 if vram >= 10 else 4
print(f'Batch size: {BATCH} (VRAM: {vram:.0f}GB)')

!python train.py \
    --data_dir ../../data/training \
    --teacher_cache ../../data/teacher_cache \
    --batch_size $BATCH \
    --num_workers 2 \
    --use_amp \
    --total_kimg 500 \
    --lr 1e-3 \
    --kd_weight 2.0 \
    --r1_gamma 10.0 \
    --log_every 50 \
    --vis_every 200 \
    --save_every 2000 \
    --output_dir ../../runs/migan_distill

In [ ]:
# Step 5: Export ONNX + download
!python export_onnx.py \
    --checkpoint ../../runs/migan_distill/checkpoints/best.pt \
    --output ../../runs/migan_inpaint.onnx

from google.colab import files
files.download('../../runs/migan_inpaint.onnx')
files.download('../../runs/migan_distill/checkpoints/best.pt')